# Day 8 — Practice Session · **SOLUTIONS**### Inheritance & Polymorphism · Python for Data Science**Prepared by Srinivasa Sai Chava**  ·  Boston University---> **Instructor copy.** Every question is followed by the answer and the reasoning.> The student copy (`Day8_Practice_Questions.ipynb`) is identical minus the answer blocks.| Part | Focus | Questions ||---|---|---|| A | Predict the output | 8 || B | Spot & fix the bug | 4 || C | Write the code | 5 || D | Challenge | 2 |

---# Part A — Predict the Output  *(8 min)*

---# Part A — Predict the Output**Teaching note:** A6 (a parent method calling the child's override) is the deepest ideahere and the one worth the most time. A4 is the most common real-world bug.

### A1. What prints?

In [ ]:
class Person:    def greet(self):        return "Hi"class Student(Person):    passprint(Student().greet())

*Your prediction:*  

> ### ✅ Answer A1> ```> Hi> ```> **Why:** `Student` defines nothing of its own, but inherits every method from `Person`.> Python looks in `Student` first, does not find `greet`, then looks in `Person` and finds it.>> The `pass` is Day 2's placeholder used properly — a child that adds nothing is still> useful because it names a type.

### A2. Which version runs?

In [ ]:
class Person:    def greet(self):        return "Hi, I am a person"class Student(Person):    def greet(self):        return "Hi, I am a student"print(Student().greet())print(Person().greet())

*Your prediction:*  

> ### ✅ Answer A2> ```> Hi, I am a student> Hi, I am a person> ```> **Why:** the child's `greet` **overrides** the parent's. Python stops at the first match> walking up the chain, so `Student`'s version wins for a `Student` object.>> `Person()` is unaffected — the parent keeps its own behaviour.

### A3. What does this print?

In [ ]:
class Person:    def greet(self):        return "Hi"class Student(Person):    def greet(self):        return super().greet() + ", I study at BU"print(Student().greet())

*Your prediction:*  

> ### ✅ Answer A3> ```> Hi, I study at BU> ```> **Why:** `super().greet()` runs the **parent's** version and returns its result, which the> child then adds to.>> **Overriding replaces; `super()` extends.** Ask the class what would happen without the> `super()` call → just `", I study at BU"`, with the parent's work thrown away.

### A4. Does this work?

In [ ]:
class Person:    def __init__(self, name):        self.name = nameclass Student(Person):    def __init__(self, name, school):        self.school = schools = Student("Ravi", "BU")print(s.school)print(s.name)

*Your prediction:*  

> ### ✅ Answer A4> ```> BU> AttributeError: 'Student' object has no attribute 'name'> ```> **Why:** writing your own `__init__` **replaces** the parent's completely. `Person.__init__`> never ran, so `self.name` was never set.>> **The fix — and the single most important line of the day:**> ```python> def __init__(self, name, school):>     super().__init__(name)      # parent's setup FIRST>     self.school = school        # then what is new> ```>> Note the failure is **delayed**: `school` works fine, and the crash only comes when> something reads `.name` — often far from the real cause.

### A5. True or False for each?

In [ ]:
class Person: passclass Student(Person): passs = Student()print(isinstance(s, Student))print(isinstance(s, Person))print(issubclass(Student, Person))print(issubclass(Person, Student))

*Your prediction:*  

> ### ✅ Answer A5> ```> True> True> True> False> ```> **Why:** a `Student` genuinely **is** a `Person` — not merely similar to one. That is what> makes the second line `True`, and it is the whole basis of polymorphism: any code written> to accept a `Person` will accept a `Student`.>> The last line is `False` because inheritance runs one way. A `Person` is not a `Student`.

### A6. Look carefully at which `pay()` runs.

In [ ]:
class Employee:    def __init__(self, salary):        self.salary = salary    def pay(self):        return self.salary    def report(self):        return f"Paid: {self.pay()}"class Intern(Employee):    def pay(self):        return 1000print(Intern(60000).report())

*Your prediction:*  

> ### ✅ Answer A6> ```> Paid: 1000> ```> **Why — and this is the deepest idea of the session:** `report()` is defined in `Employee`> and is not overridden. But inside it, `self.pay()` is looked up on **the actual object**,> which is an `Intern`. So the child's `pay()` runs, even though the calling method lives> in the parent.>> **A parent method calling `self.something()` gets the child's override.** This is> polymorphism working *inside* the parent class, and it is how frameworks let you customise> behaviour by overriding one method.>> Students often predict `60000` here. Walk it slowly.

### A7. Which parent wins?

In [ ]:
class Swimmer:    def move(self): return "swims"class Flyer:    def move(self): return "flies"class Duck(Swimmer, Flyer):    passprint(Duck().move())

*Your prediction:*  

> ### ✅ Answer A7> ```> swims> ```> **Why:** with multiple parents, Python searches **left to right** and stops at the first> match. `Swimmer` is listed first, so its `move()` wins.>> You can always inspect the order:> ```python> Duck.__mro__> (Duck, Swimmer, Flyer, object)> ```>> Swap the class list to `Duck(Flyer, Swimmer)` and the answer becomes `"flies"`.

### A8. Neither class has a parent. Does the loop work?

In [ ]:
class Dog:    def speak(self): return "Woof"class Robot:    def speak(self): return "Beep"for x in [Dog(), Robot()]:    print(x.speak())

*Your prediction:*  

> ### ✅ Answer A8> ```> Woof> Beep> ```> **Why — duck typing.** Python never checks an object's type before calling a method.> It simply tries. Both objects have a `speak()` method, so both calls work — the family> tree is irrelevant.>> *"If it walks like a duck and quacks like a duck..."*>> This is why `len()` works on a string, a list, a dict and your own class: anything that> defines `__len__`. **Polymorphism does not require inheritance in Python.**

---# Part B — Spot & Fix the Bug  *(7 min)*

---# Part B — Spot & Fix the Bug**Teaching note:** B1 and B2 are the same bug with different symptoms — one crashes, onesilently loses data. B2 is the more dangerous of the two.

### B1. This crashes when reading the name.

In [ ]:
class Animal:    def __init__(self, name):        self.name = nameclass Dog(Animal):    def __init__(self, name, breed):        self.breed = breedd = Dog("Rex", "Labrador")print(d.name)

*Error:* *Your fix:*

> ### ✅ Answer B1> **Error:** `AttributeError: 'Dog' object has no attribute 'name'`> **Cause:** the child's `__init__` replaced the parent's, so `Animal.__init__` never ran> and `self.name` was never set.

In [ ]:
class Animal:    def __init__(self, name):        self.name = nameclass Dog(Animal):    def __init__(self, name, breed):        super().__init__(name)      # THE FIX - parent setup first        self.breed = breedd = Dog("Rex", "Labrador")print(d.name, "|", d.breed)

### B2. No crash this time — but something is still wrong.

In [ ]:
class Animal:    def __init__(self, name):        self.name = name        self.legs = 4class Bird(Animal):    def __init__(self, name):        self.legs = 2                 # birds have two        super().__init__(name)        # called LASTb = Bird("Tweety")print(b.name, b.legs)

*What's wrong:* *Your fix:*

> ### ✅ Answer B2> **Symptom:** prints `Tweety 4` — the bird has four legs.> **Cause:** `super().__init__()` was called **last**, so the parent's `self.legs = 4`> overwrote the child's `self.legs = 2`.>> No error, just silently wrong data — which makes this more dangerous than B1.

In [ ]:
class Animal:    def __init__(self, name):        self.name = name        self.legs = 4class Bird(Animal):    def __init__(self, name):        super().__init__(name)        # parent FIRST        self.legs = 2                 # then override itb = Bird("Tweety")print(b.name, b.legs)                 # Tweety 2# THE RULE: call super().__init__() first, then set what is different.

### B3. Predict the error.

In [ ]:
class Person:    def __init__(self, name):        self.name = nameclass Student(Person):    def __init__(self, name, school):        Person.__init__(name)        self.school = schools = Student("Ravi", "BU")

*Error:* *Your fix:*

> ### ✅ Answer B3> **Error:** `TypeError: Person.__init__() missing 1 required positional argument: 'name'`> **Cause:** calling the parent through the **class** rather than through `super()` means> Python does not fill in `self` for you. `name` landed in the `self` slot, leaving nothing> for `name`.

In [ ]:
class Person:    def __init__(self, name):        self.name = nameclass Student(Person):    def __init__(self, name, school):        super().__init__(name)         # super() supplies self automatically        self.school = schools = Student("Ravi", "BU")print(s.name, s.school)# The class-based form also works, but you must pass self yourself:class Student2(Person):    def __init__(self, name, school):        Person.__init__(self, name)    # note the explicit self        self.school = schoolprint(Student2("Sara", "BU").name)# Prefer super() - it is shorter and keeps working if the parent changes.

### B4. The child method will not run. Why?

In [ ]:
class Animal:    def speak(self):        return "..."class Dog(Animal):    def speak():        return "Woof"print(Dog().speak())

*Error:* *Your fix:*

> ### ✅ Answer B4> **Error:** `TypeError: Dog.speak() takes 0 positional arguments but 1 was given`> **Cause:** the child's `speak` is missing `self`. Python still passes the object, so> there is one argument for a method that declared no slots.>> Day 7's rule has not changed just because we are inheriting: **every method takes `self`> first.**

In [ ]:
class Animal:    def speak(self):        return "..."class Dog(Animal):    def speak(self):          # self, always        return "Woof"print(Dog().speak())

---# Part C — Write the Code  *(12 min)*

---# Part C — Write the Code**Teaching note:** C2 is the one to demonstrate — extending with `super()` rather thanreplacing. C5 is the composition question and the most conceptually valuable.

### C1. A `Shape` hierarchyWrite a `Shape` parent with a `name` attribute and an `area()` method returning `0`.Then write `Circle` (radius) and `Square` (side) children that each override `area()`.Print the area of one of each.

In [ ]:
# your code here

In [ ]:
class Shape:    """Any two-dimensional shape."""    def __init__(self, name):        self.name = name    def area(self):        return 0                       # children override thisclass Circle(Shape):    def __init__(self, radius):        super().__init__("Circle")     # parent setup first        self.radius = radius    def area(self):        return 3.14159 * self.radius ** 2class Square(Shape):    def __init__(self, side):        super().__init__("Square")        self.side = side    def area(self):        return self.side ** 2for s in [Circle(2), Square(3)]:    print(f"{s.name}: {s.area():.2f}")# Circle: 12.57# Square: 9.00

### C2. Extend, do not replaceWrite an `Employee` with `name` and `salary`, and a `pay()` returning the salary.Then write a `Manager` child that takes an extra `bonus` and whose `pay()` returns**salary + bonus** — by calling the parent's `pay()` with `super()`, not by recalculating.

In [ ]:
# your code here

In [ ]:
class Employee:    def __init__(self, name, salary):        self.name   = name        self.salary = salary    def pay(self):        return self.salaryclass Manager(Employee):    def __init__(self, name, salary, bonus):        super().__init__(name, salary)      # parent first        self.bonus = bonus    def pay(self):        return super().pay() + self.bonus   # EXTEND, do not recalculatee = Employee("Ravi", 50000)m = Manager("Sara", 60000, 5000)print(e.name, e.pay())      # Ravi 50000print(m.name, m.pay())      # Sara 65000# Why super().pay() instead of  return self.salary + self.bonus ?# If the pay formula ever changes in Employee, Manager updates for free.# Recalculating would silently drift out of step.

### C3. Polymorphism in a loopWrite three classes — `Car`, `Bicycle`, `Boat` — each with a `travel()` method returning adifferent sentence. Put one of each in a list and print them all with a single loop.They do **not** need a common parent — show that duck typing is enough.

In [ ]:
# your code here

In [ ]:
class Car:    def travel(self): return "drives on roads"class Bicycle:    def travel(self): return "is pedalled along"class Boat:    def travel(self): return "sails on water"for vehicle in [Car(), Bicycle(), Boat()]:    print(f"{type(vehicle).__name__:9} {vehicle.travel()}")# No parent class anywhere - duck typing is enough for polymorphism.# Adding a Plane class needs no change to this loop at all.

### C4. A `SavingsAccount`Extend Day 7's `BankAccount` (given below) into a `SavingsAccount` that:- takes an extra `rate` (as a decimal, e.g. `0.05`)- has an `add_interest()` method that deposits `balance × rate`

In [ ]:
class BankAccount:    def __init__(self, owner, balance=0):        self.owner   = owner        self.balance = balance    def deposit(self, amount):        self.balance += amount    def __str__(self):        return f"{self.owner}: {self.balance:.2f}"# your SavingsAccount here

In [ ]:
class BankAccount:    def __init__(self, owner, balance=0):        self.owner   = owner        self.balance = balance    def deposit(self, amount):        self.balance += amount    def __str__(self):        return f"{self.owner}: {self.balance:.2f}"class SavingsAccount(BankAccount):    """A bank account that earns interest."""    def __init__(self, owner, balance=0, rate=0.05):        super().__init__(owner, balance)     # parent setup first        self.rate = rate    def add_interest(self):        self.deposit(self.balance * self.rate)   # reuse the inherited methods = SavingsAccount("Ravi", 1000, 0.05)print(s)                    # Ravi: 1000.00s.add_interest()print(s)                    # Ravi: 1050.00s.deposit(200)              # inherited from BankAccountprint(s)                    # Ravi: 1250.00# add_interest calls self.deposit() - the INHERITED method. The child does not# need to know how depositing works, only that it does.

### C5. Inheritance or composition?For each pair below, say whether the second should **inherit from** the first,or **contain** one. Then write the correct version of the third pair.1. `Animal` and `Dog`2. `Engine` and `Car`3. `Person` and `Address`

In [ ]:
# your answers and code here

In [ ]:
# 1. Animal / Dog        -> INHERIT.  "A Dog IS AN Animal" is true.# 2. Engine / Car        -> CONTAIN.  "A Car IS AN Engine" is false; it HAS one.# 3. Person / Address    -> CONTAIN.  "A Person IS AN Address" is nonsense.class Address:    def __init__(self, street, city):        self.street = street        self.city   = city    def __str__(self):        return f"{self.street}, {self.city}"class Person:    def __init__(self, name, address):        self.name    = name        self.address = address        # HAS AN address    def __str__(self):        return f"{self.name} — {self.address}"addr = Address("12 MG Road", "Pune")p    = Person("Ravi", addr)print(p)# THE TEST: say it out loud. If "X IS A Y" sounds wrong to an ordinary# English speaker, the inheritance is wrong too - however convenient the# code reuse would have been.

---# Part D — Challenge  *(3 min, or take home)*

---# Part D — Challenge**Teaching note:** D1 is the framework pattern in miniature — worth showing because it ishow every library they will use is built.

### D1. A parent that depends on its childrenWrite a `Report` parent with:- a `title()` method returning `"Untitled"`- a `body()` method returning `""`- a `render()` method that returns `title()` and `body()` joined by a newlineThen write two children that override only `title()` and `body()` — **not** `render()` —and show that `render()` picks up each child's versions.

In [ ]:
# your code here

In [ ]:
class Report:    """A report skeleton. Children fill in the parts."""    def title(self):        return "Untitled"    def body(self):        return ""    def render(self):                    # children NEVER override this        return f"{self.title()}\n{self.body()}"class SalesReport(Report):    def title(self): return "SALES"    def body(self):  return "Revenue up 12%"class HRReport(Report):    def title(self): return "HR"    def body(self):  return "3 new hires"for r in [SalesReport(), HRReport(), Report()]:    print(r.render())    print("-" * 20)# render() lives in the PARENT and was never overridden - yet it produces# different output for each child, because self.title() and self.body() are# looked up on the actual object.## This is the framework pattern: the parent controls the overall shape,# children supply the pieces. Every web framework and plugin system you# will ever use works this way.

### D2. Where does the method come from?Given the classes below, predict what each call prints — then check with `__mro__`.

In [ ]:
class A:    def who(self): return "A"class B(A):    def who(self): return "B"class C(A):    def who(self): return "C"class D(B, C):    pass# Predict, then run:# print(D().who())# print(D.__mro__)

*Your prediction:*  

In [ ]:
class A:    def who(self): return "A"class B(A):    def who(self): return "B"class C(A):    def who(self): return "C"class D(B, C):    passprint(D().who())        # "B"print(D.__mro__)# (D, B, C, A, object)

> ### ✅ Answer D2> ```> B> (<class 'D'>, <class 'B'>, <class 'C'>, <class 'A'>, <class 'object'>)> ```> **Why:** this is the classic **diamond**: `D` inherits from `B` and `C`, which both> inherit from `A`. Python flattens that into a single order — `D → B → C → A → object` —> and stops at the first `who()` it finds, which is `B`'s.>> Notice that `A` appears **once**, after both `B` and `C`. Python guarantees a parent> always comes after all of its children in the order, which is what stops `A`'s version> shadowing `C`'s.>> **The teaching point is not the algorithm.** It is that answering *"which method runs?"*> already required drawing a diagram — and a reader of your code would have to do the same.> That is the argument for keeping to single inheritance in your own work.

---## Done? Self-check- [ ] I can write a child class that reuses a parent- [ ] I know why a child `__init__` must call `super().__init__()` — and call it **first**- [ ] I know the difference between overriding and extending with `super()`- [ ] I know why `isinstance(student, Person)` is `True`- [ ] I can explain why a parent method calling `self.x()` gets the child's version- [ ] I know that polymorphism does not require inheritance in Python- [ ] I can apply the "IS A / HAS A" test before inheriting### Homework1. Build a `Shape` parent with `Circle` and `Square` children, each with `area()`.2. Extend Day 7's `BankAccount` into a `SavingsAccount` that adds interest.3. Find one place in your own code where inheritance would be the wrong choice.### Next class — Topic 1.9: File I/OReading and writing text and CSV files, the `with` statement, and file modes.---*Slides & notebooks by Srinivasa Sai Chava · Boston University*

---## Wrap-up — running the last 5 minutesThree cold-call questions:1. *"My child object has no `name` attribute. What did I forget?"* → `super().__init__()`2. *"Overriding versus `super()` — what is the difference?"* → replace versus extend.3. *"Does polymorphism need a parent class in Python?"* → no, duck typing is enough.**Common misconceptions to watch for today**| Misconception | Correction ||---|---|| "The child automatically runs the parent's `__init__`" | Only if the child has no `__init__` of its own || "`super().__init__()` can go anywhere" | Call it first, or it overwrites what you just set || "`Person.__init__(name)` is the same as `super().__init__(name)`" | The class form needs an explicit `self` || "Overriding and extending are the same" | Overriding replaces; `super()` keeps the parent's work || "A parent method always uses the parent's other methods" | `self.x()` is looked up on the actual object || "Polymorphism requires inheritance" | Duck typing needs only a shared method name || "Inheritance is for reusing code" | It is for "IS A". Reuse alone → composition |**The two deepest points**, worth revisiting if time allows: A6 (a parent method picking upthe child's override) and D1 (the framework pattern built on exactly that). Together theyexplain how every library the students will import is structured.**Homework given:** `Shape` hierarchy; `SavingsAccount`; find a bad inheritance case.**Next session:** Topic 1.9 — File I/O. OOP is now complete; from tomorrow they startgetting real data in and out of files, which is the last step before NumPy and pandas.